In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
num_blocks = 30
blocks = np.arange(1, num_blocks + 1)

# Patch configurations
configs = [
    {"patch": 4, "token_deg": 1.0, "windows": [2, 3, 5]},
    {"patch": 2, "token_deg": 0.5, "windows": [2, 3, 5]},
]

plt.figure(figsize=(10, 6))

for cfg in configs:
    token_deg = cfg["token_deg"]
    patch = cfg["patch"]

    for w in cfg["windows"]:
        # One interaction every 2 blocks (shifted window)
        interactions = blocks // 2

        # Radius growth
        step_radius = (w - 1) / 2
        radius_deg = interactions * step_radius * token_deg

        label = f"{patch}x{patch} patch, {w}x{w} window"
        plt.plot(blocks, radius_deg, label=label)

plt.xlabel("Swin block index")
plt.ylabel("Effective propagation radius (degrees)")
plt.title("Effective spatial propagation vs depth")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --------------------
# Global parameters
# --------------------
num_blocks = 30
blocks = np.arange(1, num_blocks + 1)
km_per_degree = 111.0

# Configurations
configs = [
    # patch, token_deg, tokens, d_scale, windows
    {"patch": 4, "token_deg": 1.0, "tokens": 720*1440//16, "d_scale": 1.0, "windows": [2, 3, 5]},
    {"patch": 2, "token_deg": 0.5, "tokens": 720*1440//4,  "d_scale": 0.5, "windows": [2, 3, 5]},
]

# --------------------
# Plot 1: Propagation radius (km)
# --------------------
plt.figure(figsize=(10, 6))

for cfg in configs:
    patch = cfg["patch"]
    token_deg = cfg["token_deg"]

    for w in cfg["windows"]:
        interactions = blocks // 2
        step_radius = (w - 1) / 2
        radius_km = interactions * step_radius * token_deg * km_per_degree

        label = f"{patch}x{patch} patch, {w}x{w} window"
        plt.plot(blocks, radius_km, label=label)

plt.xlabel("Swin block index")
plt.ylabel("Effective propagation radius (km)")
plt.title("Effective spatial propagation vs depth")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# --------------------
# Plot 2: Relative FLOPs
# --------------------
plt.figure(figsize=(10, 6))

for cfg in configs:
    patch = cfg["patch"]
    N = cfg["tokens"]
    d_scale = cfg["d_scale"]

    for w in cfg["windows"]:
        flops_per_block = N * (w ** 2) * d_scale
        flops = flops_per_block * blocks

        label = f"{patch}x{patch} patch, {w}x{w} window"
        plt.plot(blocks, flops / 1e9, label=label)

plt.xlabel("Swin block index")
plt.ylabel("Relative attention FLOPs (×10⁹ units)")
plt.title("Cumulative attention FLOPs vs depth")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Parameters
blocks = 30  # Total blocks
token_size_deg = 0.5  # Each token represents 0.5° (patch size 2x2)

# Even window sizes to compare
sWins = [2, 4]  # 2x2, 4x4 windows (even-sized)

# Patch sizes
patch_sizes = [2, 4]  # 2x2 and 4x4 patches (in pixels)

# Calculate effective propagation radius per block
def calculate_effective_radius(patch_size, sWin, blocks):
    # Calculate number of tokens for given patch size
    tokens_per_patch = 720 // patch_size  # For latitude (assuming square grid, 720x1440)
    
    # Calculate radius (in tokens) after L blocks
    radius_tokens = (blocks // 2) * (sWin - 1)
    
    # Convert to degrees (each token = 0.5°)
    radius_deg = radius_tokens * token_size_deg
    
    # Calculate total propagation radius for each block
    total_radius = [radius_deg * (i + 1) for i in range(blocks)]
    
    return total_radius

# Create plots for each window size
plt.figure(figsize=(10, 6))

for patch_size in patch_sizes:
    for sWin in sWins:
        total_radius = calculate_effective_radius(patch_size, sWin, blocks)
        label = f'Patch {patch_size}x{patch_size}, Window {sWin}x{sWin}'
        plt.plot(range(1, blocks + 1), total_radius, label=label)

# Labels and legend
plt.title('Effective Propagation Radius over 30 Blocks (Even-sized Windows Only)')
plt.xlabel('Block Index')
plt.ylabel('Propagation Radius (degrees)')
plt.legend()
plt.grid(True)
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
import numpy as np

# Input resolution
H, W = 720, 1440
blocks = 30

# Patch and window configurations
patch_sizes = [2, 4]       # pixels
sWins = [2, 4]      # tokens

# Assume embedding dimension proportional to patch area
def embedding_dim(p):
    return p**2  # simple proportionality

def compute_flops(H, W, p, w, blocks):
    num_patches_H = H // p
    num_patches_W = W // p
    num_windows_H = num_patches_H // w
    num_windows_W = num_patches_W // w
    num_windows = num_windows_H * num_windows_W
    d = embedding_dim(p)
    flops_per_window = w**2 * w**2 * d
    total_flops = flops_per_window * num_windows * blocks
    return total_flops

# Generate table
print(f"{'Patch':>5} {'Window':>6} {'FLOPs (relative)':>20}")
flops_list = []
for p in patch_sizes:
    for w in sWins:
        flops = compute_flops(H, W, p, w, blocks)
        flops_list.append(flops)
        print(f"{p:5} {w:6} {flops:20,.0f}")

# Optional: normalize relative to smallest config
flops_array = np.array(flops_list)
rel_flops = flops_array / flops_array.min()
print("\nRelative FLOPs (normalized to smallest config):")
print(rel_flops)


In [ ]:
import numpy as np

# Input resolution
H, W = 720, 1440
blocks = 30

# Patch sizes to compare
patch_sizes = [2, 4]  # pixels

# Window size schedule (even blocks: w, odd blocks: w//2)
window_schedules = {
    2: [2, 1],   # Example: base window 2x2 -> alternate 2/1
    4: [4, 2]    # base window 4x4 -> alternate 4/2
}

# Embedding dimension proportional to patch area
def embedding_dim(p):
    return p**2  # simple proportionality

# Compute FLOPs with window schedule
def compute_flops_schedule(H, W, p, window_schedule, blocks):
    d = embedding_dim(p)
    total_flops = 0
    for b in range(blocks):
        w = window_schedule[b % len(window_schedule)]
        num_patches_H = H // p
        num_patches_W = W // p
        num_windows_H = num_patches_H // w
        num_windows_W = num_patches_W // w
        num_windows = num_windows_H * num_windows_W
        flops_per_window = w**2 * w**2 * d
        total_flops += flops_per_window * num_windows
    return total_flops

# Generate table
print(f"{'Patch':>5} {'Base Win':>8} {'FLOPs (relative)':>20}")
flops_list = []
for p in patch_sizes:
    for base_w, schedule in window_schedules.items():
        flops = compute_flops_schedule(H, W, p, schedule, blocks)
        flops_list.append(flops)
        print(f"{p:5} {base_w:8} {flops:20,.0f}")

# Optional: relative FLOPs
flops_array = np.array(flops_list)
rel_flops = flops_array / flops_array.min()
print("\nRelative FLOPs (normalized to smallest config):")
print(rel_flops)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Input resolution
H, W = 720, 1440
blocks = 30

# Patch/window options
patch_sizes = [2, 4]
sWins = [2, 4]

# Embedding dimension ~ patch area
def embedding_dim(p):
    return p**2

# FLOPs per config
def compute_flops(H, W, p, w, blocks):
    d = embedding_dim(p)
    num_patches_H = H // p
    num_patches_W = W // p
    num_windows_H = num_patches_H // w
    num_windows_W = num_patches_W // w
    num_windows = num_windows_H * num_windows_W
    flops_per_window = w**4 * d
    total_flops = flops_per_window * num_windows * blocks
    return total_flops

# Effective propagation radius
token_size_deg = 1  # each patch = 4x4 pixels -> 1 degree
def effective_radius(p, w, blocks):
    return (w/2 * token_size_deg) * blocks

# Prepare data
configs = []
params = []
flops = []
radius = []

for p in patch_sizes:
    for w in sWins:
        configs.append(f"P{p}W{w}")
        params.append(embedding_dim(p))
        flops.append(compute_flops(H, W, p, w, blocks))
        radius.append(effective_radius(p, w, blocks))

# Normalize for plotting
params_norm = np.array(params)/max(params)
flops_norm = np.array(flops)/max(flops)
radius_norm = np.array(radius)/max(radius)

# Plot
x = np.arange(len(configs))
width = 0.25

fig, ax = plt.subplots(figsize=(10,6))
ax.bar(x - width, params_norm, width, label='Parameters (norm)')
ax.bar(x, flops_norm, width, label='FLOPs (norm)')
ax.bar(x + width, radius_norm, width, label='Propagation radius (norm)')

# Highlight Patch 4x4 + Window 2x2
highlight_idx = configs.index("P4W2")
ax.bar(x[highlight_idx] - width, params_norm[highlight_idx], width, color='orange')
ax.bar(x[highlight_idx], flops_norm[highlight_idx], width, color='orange')
ax.bar(x[highlight_idx] + width, radius_norm[highlight_idx], width, color='orange')

ax.set_xticks(x)
ax.set_xticklabels(configs)
ax.set_ylabel("Normalized value")
ax.set_title("Patch/Window Configurations: Params, FLOPs, Propagation Radius")
ax.legend()
ax.grid(True)

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Input resolution
H, W = 720, 1440
blocks = 30

# Patch/window options
patch_sizes = [2, 4]
sWins = [2, 4]

# Embedding dimension ~ patch area
def embedding_dim(p):
    return p**2

# FLOPs per config
def compute_flops(H, W, p, w, blocks):
    d = embedding_dim(p)
    num_patches_H = H // p
    num_patches_W = W // p
    num_windows_H = num_patches_H // w
    num_windows_W = num_patches_W // w
    num_windows = num_windows_H * num_windows_W
    flops_per_window = w**4 * d
    total_flops = flops_per_window * num_windows * blocks
    return total_flops

# Effective propagation radius
token_size_deg = 1  # each patch = 4x4 pixels -> 1 degree
def effective_radius(p, w, blocks):
    return (w/2 * token_size_deg) * blocks

# Prepare data
configs = []
params = []
flops = []
radius = []

for p in patch_sizes:
    for w in sWins:
        configs.append(f"P{p}W{w}")
        params.append(embedding_dim(p))
        flops.append(compute_flops(H, W, p, w, blocks))
        radius.append(effective_radius(p, w, blocks))

# Normalize for plotting
params_norm = np.array(params)/max(params)
flops_norm = np.array(flops)/max(flops)
radius_norm = np.array(radius)/max(radius)

# Plot
x = np.arange(len(configs))
width = 0.25

fig, ax = plt.subplots(figsize=(10,6))

colors = ['skyblue', 'lightgreen', 'salmon']  # default colors
highlight_colors = ['dodgerblue', 'green', 'red']  # for P4W2

for i in range(len(configs)):
    c_params = highlight_colors[0] if configs[i]=="P4W2" else colors[0]
    c_flops  = highlight_colors[1] if configs[i]=="P4W2" else colors[1]
    c_radius = highlight_colors[2] if configs[i]=="P4W2" else colors[2]
    ax.bar(x[i] - width, params_norm[i], width, color=c_params)
    ax.bar(x[i], flops_norm[i], width, color=c_flops)
    ax.bar(x[i] + width, radius_norm[i], width, color=c_radius)

ax.set_xticks(x)
ax.set_xticklabels(configs)
ax.set_ylabel("Normalized value")
ax.set_title("Patch/Window Configurations: Params, FLOPs, Propagation Radius")
ax.legend(["Parameters", "FLOPs", "Propagation radius"])
ax.grid(True)

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Patch sizes (pixels)
patch_sizes = np.array([2, 4, 8])  # you can extend if needed

# Embedding dimension proportional to patch area
embedding_dim = patch_sizes ** 2

# Parameters in attention/MLP ∝ d^2
params = embedding_dim ** 2

# Plot
fig, ax1 = plt.subplots(figsize=(8,5))

color1 = 'tab:blue'
ax1.set_xlabel('Patch size (pixels)')
ax1.set_ylabel('Embedding dimension', color=color1)
ax1.plot(patch_sizes, embedding_dim, 'o-', color=color1, label='Embedding dimension (d)')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xticks(patch_sizes)

ax2 = ax1.twinx()  # share x-axis
color2 = 'tab:red'
ax2.set_ylabel('Parameter scaling (d²)', color=color2)
ax2.plot(patch_sizes, params, 's--', color=color2, label='Parameter scaling (d²)')
ax2.tick_params(axis='y', labelcolor=color2)

fig.suptitle('Embedding Dimension and Parameter Scaling vs Patch Size')
fig.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Patch sizes (pixels)
patch_sizes = np.array([2, 4, 8])  # extendable
embedding_dim = patch_sizes ** 2
params = embedding_dim ** 2

# Highlight optimal patch
optimal_patch = 4
optimal_window = 2  # just for labeling

# Plot
fig, ax1 = plt.subplots(figsize=(8,5))

color1 = 'tab:blue'
ax1.set_xlabel('Patch size (pixels)')
ax1.set_ylabel('Embedding dimension', color=color1)
ax1.plot(patch_sizes, embedding_dim, 'o-', color=color1, label='Embedding dimension (d)')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xticks(patch_sizes)

# Mark optimal patch
opt_idx = np.where(patch_sizes==optimal_patch)[0][0]
ax1.plot(optimal_patch, embedding_dim[opt_idx], 'o', color='orange', markersize=12, label=f'Optimal P{optimal_patch}W{optimal_window}')

ax2 = ax1.twinx()
color2 = 'tab:red'
ax2.set_ylabel('Parameter scaling (d²)', color=color2)
ax2.plot(patch_sizes, params, 's--', color=color2, label='Parameter scaling (d²)')
ax2.tick_params(axis='y', labelcolor=color2)
ax2.plot(optimal_patch, params[opt_idx], 's', color='orange', markersize=12)

fig.suptitle('Embedding Dimension and Parameter Scaling vs Patch Size\nOptimal Choice: P4W2')
fig.tight_layout()
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Settings
blocks = 30
patch_size = 4       # 4x4 patch → 1° per patch
initial_window = 2   # 2x2 window

# Window schedule: first 30 blocks all 2x2 (can change later)
window_schedule = [2]*blocks
# Example for mixed schedule (last 10 blocks larger window)
# window_schedule = [2]*20 + [4]*10

# Effective radius per block
# radius per block = window_radius = sWin / 2 * patch_size in degrees
radius_per_block = [w/2 * 1 for w in window_schedule]  # 1° per patch

# Cumulative effective radius
cum_radius = np.cumsum(radius_per_block)

# Plot
plt.figure(figsize=(12,5))
plt.plot(range(1, blocks+1), cum_radius, 'o-', label='Effective propagation radius (°)')
plt.bar(range(1, blocks+1), window_schedule, alpha=0.3, color='orange', label='Window size (patches)')

plt.xlabel("Block index")
plt.ylabel("Effective propagation radius (°) / Window size (patches)")
plt.title("Swin Transformer Blocks: Effective Propagation & Window Schedule (Patch 4x4)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Settings
blocks = 30
patch_size = 4  # 4x4 patch -> 1° per patch

# Window schedules
window_fixed = [2]*blocks                # Fixed small window
window_increasing = [2]*20 + [4]*10      # Increase last 10 blocks

def cumulative_radius(window_schedule):
    # Effective radius per block = window_radius = sWin / 2 * patch_size (deg)
    radius_per_block = [w/2 * 1 for w in window_schedule]  # 1° per patch
    return np.cumsum(radius_per_block)

# Compute cumulative propagation radius
radius_fixed = cumulative_radius(window_fixed)
radius_inc = cumulative_radius(window_increasing)

# Plot
plt.figure(figsize=(12,5))
plt.plot(range(1, blocks+1), radius_fixed, 'o-', label='Fixed window 2x2', color='blue')
plt.plot(range(1, blocks+1), radius_inc, 's--', label='Increasing window 2→4', color='red')

# Optional: overlay window size as bars
plt.bar(range(1, blocks+1), window_fixed, alpha=0.2, color='blue', label='Window size fixed')
plt.bar(range(1, blocks+1), window_increasing, alpha=0.2, color='red', label='Window size increasing')

plt.xlabel("Block index")
plt.ylabel("Cumulative effective propagation radius (°)")
plt.title("Propagation Radius over 30 Swin Transformer Blocks (Patch 4x4)")
plt.legend()
plt.grid(True)
plt.show()


In [2]:
from Zeus_Swin_Block_3D_CDHW import PositionBiasLearner3D

pos_bias_learner = PositionBiasLearner3D((14, 180, 360), (2, 4, 4), 2, 10, (2, 2, 2), 5)


In [6]:
import torch

def compute_proxy_pair_bias_batched(lat, lon, depth):
    """
    lat, lon, depth: (D_window, H_window, W_window)
    Returns: proxy_bias: (vWin, vWin, n_channels)
    """
    P = lat.numel()
    A_lat = lat.view(-1, 1)
    B_lat = lat.view(1, -1)
    A_lon, B_lon = lon.view(-1, 1), lon.view(1, -1)
    A_depth, B_depth = depth.view(-1, 1), depth.view(1, -1)

    dlat = torch.deg2rad(A_lat - B_lat)
    dlon = torch.deg2rad(((A_lon - B_lon + 180) % 360) - 180)
    cos_lat1, cos_lat2 = torch.cos(torch.deg2rad(A_lat)), torch.cos(torch.deg2rad(B_lat))
    great_circle = 2 * torch.asin(torch.sqrt(torch.sin(dlat/2)**2 + cos_lat1*cos_lat2*torch.sin(dlon/2)**2))
    delta_depth = A_depth - B_depth
    abs_delta_depth = torch.abs(delta_depth)

    proxy_bias = torch.stack([cos_lat1**2, cos_lat2**2, great_circle, delta_depth, abs_delta_depth], dim=-1)
    return proxy_bias  # shape: (vWin, vWin, 5)


def compute_all_window_correlations(pos_bias, lat, lon, depth, sWin):
    """
    pos_bias: (nWin, vWin, vWin, num_heads)
    lat/lon/depth: full-resolution (D, H, W)
    sWin: (Pz, Ph, Pw)
    
    Returns: correlations: (num_heads, n_channels) averaged over all windows
    """
    Pz, Ph, Pw = sWin
    nHeads = pos_bias.shape[-1]
    nChannels = 5
    correlations_sum = torch.zeros(nHeads, nChannels)
    nWin = pos_bias.shape[0]

    for w_idx in range(nWin):
        # Compute window coordinates in lat/lon/depth
        z = (w_idx // ((lat.shape[1]//Ph)*(lat.shape[2]//Pw))) % (lat.shape[0]//Pz)
        y = (w_idx // (lat.shape[2]//Pw)) % (lat.shape[1]//Ph)
        x = w_idx % (lat.shape[2]//Pw)

        lat_win = lat[z*Pz:(z+1)*Pz, y*Ph:(y+1)*Ph, x*Pw:(x+1)*Pw]
        lon_win = lon[z*Pz:(z+1)*Pz, y*Ph:(y+1)*Ph, x*Pw:(x+1)*Pw]
        depth_win = depth[z*Pz:(z+1)*Pz, y*Ph:(y+1)*Ph, x*Pw:(x+1)*Pw]

        proxy_bias_win = compute_proxy_pair_bias_batched(lat_win, lon_win, depth_win)
        pos_bias_win = pos_bias[w_idx]

        # Flatten for correlation
        a = pos_bias_win.view(-1, nHeads)  # (vWin^2, num_heads)
        b = proxy_bias_win.view(-1, nChannels)  # (vWin^2, n_channels)

        # Compute correlation per head/channel
        a_mean = a.mean(dim=0, keepdim=True)
        b_mean = b.mean(dim=0, keepdim=True)
        cov = ((a - a_mean).T @ (b - b_mean)) / a.shape[0]  # (num_heads, n_channels)
        std = a.std(dim=0, unbiased=False).view(-1,1) * b.std(dim=0, unbiased=False).view(1,-1) + 1e-8
        correlations_sum += cov / std

    correlations_avg = correlations_sum / nWin
    return correlations_avg


# Example usage:
pos_bias_all = pos_bias_learner.forward()  # (nWin, vWin, vWin, num_heads)
corr_avg = compute_all_window_correlations(
    pos_bias_all,
    pos_bias_learner.lat,
    pos_bias_learner.lon,
    pos_bias_learner.depth,
    pos_bias_learner.sWin
)
print("Averaged Pearson correlations (num_heads x proxy_channels):")
print(corr_avg)


RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.